# 01 — Data Ingestion

Walks `data/raw/`, parses every Polar Flow CSV, and writes a unified
`master_long.parquet` to `data/processed/`.

**Run this notebook first**, before any other notebook in the pipeline.

In [ ]:
import sys, re, logging
from pathlib import Path

import pandas as pd
import yaml

# ── path setup ────────────────────────────────────────────────────────────────
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.polar_parser import parse_polar_csv

RAW_DIR       = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = REPO_ROOT / 'config' / 'participants.yaml'

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger('01_ingestion')

print(f'RAW_DIR       : {RAW_DIR}')
print(f'PROCESSED_DIR : {PROCESSED_DIR}')

In [ ]:
# ── Load participant registry ─────────────────────────────────────────────────
with open(CONFIG_PATH, encoding='utf-8') as fh:
    config = yaml.safe_load(fh)

participant_registry = {p['folder']: p for p in config.get('participants', [])}
print(f'Registered participants: {list(participant_registry.keys())}')

In [ ]:
# ── Discover all CSVs recursively ─────────────────────────────────────────────
csv_files = sorted(RAW_DIR.rglob('*.csv'))
# Exclude .gitkeep files and example placeholders if needed
csv_files = [f for f in csv_files if not f.name.startswith('.')]

print(f'Found {len(csv_files)} CSV file(s):')
for f in csv_files:
    print(f'  {f.relative_to(REPO_ROOT)}')

In [ ]:
# ── Date extraction helper ────────────────────────────────────────────────────
_DATE_RE = re.compile(r'(\d{4}-\d{2}-\d{2})')

def extract_date(path: Path):
    """Return datetime.date from filename, or None if not parseable."""
    m = _DATE_RE.search(path.stem)
    if m:
        try:
            return pd.to_datetime(m.group(1)).date()
        except Exception:
            return None
    return None

DAY_NAMES = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

In [ ]:
# ── Main ingestion loop ───────────────────────────────────────────────────────
all_frames = []
errors = []

for csv_path in csv_files:
    user_id = csv_path.parent.name          # folder name = User_ID
    date    = extract_date(csv_path)

    result = parse_polar_csv(csv_path)
    ts = result['timeseries']
    meta = result['metadata']

    if ts.empty:
        logger.warning('Skipping %s — empty time-series', csv_path.name)
        errors.append(str(csv_path))
        continue

    # Attach identifiers
    ts = ts.copy()
    ts['User_ID']     = user_id
    ts['Date']        = pd.to_datetime(date) if date else pd.NaT
    ts['Day_of_Week'] = pd.to_datetime(date).day_name() if date else None

    # Attach scalar metadata columns (calories etc.) from the header block
    for key, val in meta.items():
        if key not in ts.columns:
            ts[key] = val

    all_frames.append(ts)
    logger.info('Loaded %-30s | rows=%d | user=%s | date=%s',
                csv_path.name, len(ts), user_id, date)

print(f'\nLoaded {len(all_frames)} session(s), {len(errors)} error(s).')
if errors:
    print('Failed files:', errors)

In [ ]:
# ── Build master_long and save ────────────────────────────────────────────────
if not all_frames:
    print('⚠️  No data loaded. Place CSV files in data/raw/<Participant_Folder>/ and re-run.')
else:
    master_long = pd.concat(all_frames, ignore_index=True)

    out_path = PROCESSED_DIR / 'master_long.parquet'
    master_long.to_parquet(out_path, index=False, engine='pyarrow')
    print(f'Saved → {out_path.relative_to(REPO_ROOT)}')
    print(f'Shape : {master_long.shape}')
    master_long.dtypes

In [ ]:
# ── Summary report ────────────────────────────────────────────────────────────
if all_frames:
    n_participants = master_long['User_ID'].nunique()
    n_days  = master_long.groupby('User_ID')['Date'].nunique().sum()
    n_total = len(master_long)

    print(f'Participants : {n_participants}')
    print(f'Total days   : {n_days}')
    print(f'Total records: {n_total:,}  (@ 1 Hz)')
    print()
    print(master_long.groupby('User_ID')['Date'].nunique().rename('sessions').to_frame())